In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz

In [42]:
tracks = pd.read_csv(r'/content/tracks.csv', keep_default_na=False)
X = load_npz('/content/text_tfidf.npz')
assert len(tracks) == X.shape[0]
assert np.array_equal(tracks.tfidf_row.to_numpy(), np.arange(X.shape[0]))
with np.load('text_tfidf.npz', allow_pickle=False) as z:
    assert np.array_equal(tracks.track_id.to_numpy(), z['track_ids'])
    terms = z['feature_names']
    idf = z['idf']
print(f'Треков с текстами: {tracks['has_lyrics'].sum()}')

Треков с текстами: 18158


In [43]:
def find_similar(track_index, k=5):
    row_idx = tracks.loc[tracks.index == track_index, 'tfidf_row'].values[0]
    if tracks.loc[tracks.index == track_index, 'has_lyrics'].values[0] == 0:
        return f'трек без текста'
    query_vector = X.getrow(row_idx)
    sim = X.dot(query_vector.T).toarray().ravel()
    sim[row_idx] = -1.0
    no_lyrics_rows = tracks[tracks['has_lyrics'] == 0]['tfidf_row'].values
    sim[no_lyrics_rows] = -1.0
    best_songs = np.argsort(sim)[::-1][:k]
    res = []
    for i in best_songs:
        sim_score = sim[i]
        track_row = tracks[tracks['tfidf_row'] == i].iloc[0]
        res.append({
            'id': track_row['track_id'],
            'name': track_row['track_name'],
            'artist': track_row['track_artist'],
            'similarity': float(sim_score),
            'lyrics': track_row['lyrics'][:150] + "..."
        })
    return res

In [44]:
valid_tracks = tracks[tracks['has_lyrics'] == 1].index.tolist()
np.random.seed(42)
test = np.random.choice(valid_tracks, size=3, replace=False)
for num in test:
    target_row = tracks.loc[num]
    print(f"Трек: '{target_row['track_name']}' - {target_row['track_artist']}")
    sim_tracks = find_similar(num, 5)
    print('Похожие треки:')
    for i, j in enumerate(sim_tracks, 1):
        print(f"{i}. '{j['name']}' - {j['artist']}, сходство: {j['similarity']:.3f}")
    print('\n')

Трек: 'Long Way Back From Hell' - Danzig
Похожие треки:
1. 'I'd Love You to Want Me' - Lobo, сходство: 0.385
2. 'Don't You Want Me' - Jody Watley, сходство: 0.357
3. 'Tell Me Do U Wanna' - Ginuwine, сходство: 0.352
4. 'Fly Away' - CRUCIFIX, сходство: 0.342
5. 'Bell Bottom Blues - 40th Anniversary Version / 2010 Remastered' - Derek & The Dominos, сходство: 0.335


Трек: 'Work It Out' - Jurassic 5
Похожие треки:
1. 'Work from Home (feat. Ty Dolla $ign)' - Fifth Harmony, сходство: 0.384
2. 'Work from Home' - Fifth Harmony, сходство: 0.384
3. 'Just Ain't Gonna Work Out' - Mayer Hawthorne, сходство: 0.366
4. 'I Like The Way You Work' - Blackstreet, сходство: 0.360
5. 'Play Hard (feat. Ne-Yo & Akon)' - David Guetta, сходство: 0.358


Трек: 'Sticky Situation' - anders
Похожие треки:
1. 'Kisha' - Lil Wayne, сходство: 0.368
2. 'PAID MY DUES' - NF, сходство: 0.341
3. 'Morning' - Teyana Taylor, сходство: 0.333
4. 'I Like The Way She Do It' - G-Unit, сходство: 0.332
5. 'Fuck It Up' - Jay Portal, с

# 1. Использованный подход

Для нахождения наиболее лексически близких треков используются TF-IDF и косинусное сходство. При подготовке рекомендаций были исключены песни без текста.

TF-IDF помогает определить, какие слова являются характерными для конкретной песни:

$TF(t, d) = f_{t, d}$

$IDF(t) = \ln\left(\dfrac{1 + N}{1 + df(t)}\right) + 1$

$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t)$

Тексты представлены в виде матрицы `text_tfidf.npz` размером $18 454 * 102 102$ (строки * слова). Для поиска лексически близких треков используется косинусное сходство, которое показывает, насколько похожи тексты по используемым словам (чем ближе значение к 1, тем выше позиция песни в рекомендациях). Так как векторы уже нормализованы, оно вычисляется с помощью скалярного произведения.

# 2. Выводы

Используемый метод позволяет искать песни со схожей лексикой, фразами и тематикой, однако может быть усовершенствован. Так как важно искать именно близкие по смыслу треки, отдельные слова могут быть не всегда информативными, не учитывается контекст и синонимия, порядок слов.

Таким образом, TF-IDF и косинусное сходство могут использоваться для базовых текстовых рекомендаций и быстрой фильтрации; для глубокого семантического анализа требуются более сложные языковые модели.